# الفصل الثالث عشر: البرمجة الكائنية - الأساس (Class, Object, __init__, self)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-13.ipynb)

> **الوقت المتوقع:** ~9 دقيقة قراءة · **12** قسمًا · **34** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **13.1** لماذا تُعتبر البرمجة الكائنية نقلة نوعية في التفكير؟
- **13.2** المفاهيم الأساسية: Class و Object
- **13.3** بناء أول Class: التشريح الكامل
- **13.4** Instance Attributes مقابل Class Attributes
- **13.5** Methods بأنواعها الثلاثة
- **13.6** Magic Methods (Dunder Methods): تخصيص سلوك الكائن
- **13.7** خصوصية البيانات: Public, Protected, Private (بأسلوب Python)
- **13.8** مشروع تطبيقي: نظام إدارة مكتبة شخصية
- **13.9** تمارين
- **13.10** اختبر فهمك (Quiz)
- **13.11** ملخص الفصل (Cheat Sheet)
- **13.12** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

## 13.1 لماذا تُعتبر البرمجة الكائنية نقلة نوعية في التفكير؟

تخيّل أنك تبني نظاماً لإدارة طلاب مدرسة. بالطريقة التي تعلمناها حتى الآن، قد تكتب:

In [ ]:
student1_name = "أحمد"
student1_age = 20
student1_grades = [85, 90, 78]

student2_name = "سارة"
student2_age = 22
student2_grades = [92, 88, 95]

بمجرد وصولك لثلاثة طلاب، ستحتاج تسعة متغيرات منفصلة! وإذا أردت دالة تحسب متوسط درجات طالب، ستحتاج تمرير ثلاثة معاملات منفصلة لكل طالب في كل مرة. الكود يصبح فوضوياً وصعب الصيانة بسرعة.

الحل الأنيق: اجعل Python «تفهم» مفهوم «طالب» ككيان واحد متكامل، يحمل بياناته الخاصة (الاسم، العمر، الدرجات) **وسلوكه** الخاص (حساب المتوسط، إضافة درجة) في **حزمة واحدة**.

In [ ]:
class Student:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age
        self.grades: list[float] = []

    def add_grade(self, grade: float) -> None:
        self.grades.append(grade)

    def average_grade(self) -> float:
        if not self.grades:
            return 0.0
        return sum(self.grades) / len(self.grades)

student1 = Student("أحمد", 20)
student1.add_grade(85)
student1.add_grade(90)
print(student1.average_grade())  # 87.5

الآن `student1` هو كائن واحد متكامل يحمل بياناته وسلوكه معاً. يمكنك إنشاء مئات الطلاب بنفس «القالب» (Class)، وكل واحد منهم مستقل تماماً عن الآخر.

هذا النموذج — **Object-Oriented Programming (OOP)** — هو الأساس الذي بُنيت عليه تقريباً كل أنظمة البرمجيات الكبرى: من تطبيقات الويب، إلى الألعاب، إلى أنظمة إدارة قواعد البيانات. حتى في Python نفسها: كل شيء تقريباً — النصوص، القوائم، القواميس — هو كائن (Object) من فئة (Class) معينة!

بنهاية هذا الفصل، ستفهم بعمق كيف تُصمم «قوالب» (Classes) لإنشاء كائنات (Objects) حقيقية في برامجك.

## 13.2 المفاهيم الأساسية: Class و Object

### الفرق بين Class و Object

- **Class (الفئة/الصنف)**: قالب أو «مخطط هندسي» يصف شكل وسلوك شيء معين، لكنه ليس الشيء نفسه.
- **Object (الكائن)**: نسخة فعلية (Instance) تم إنشاؤها من القالب، ولها بيانات حقيقية خاصة بها.

> 🔄 **تشبيه عملي**: الـ Class أشبه بـ«مخطط بناء منزل» يحدد: عدد الغرف، شكل النوافذ، التصميم العام. الـ Object هو «المنزل الفعلي» المبني من هذا المخطط — يمكنك بناء عشرات المنازل من نفس المخطط، وكل منزل (Object) مستقل، يمكن طلاؤه بلون مختلف وتأثيثه بأثاث مختلف، رغم أنه بُني من نفس التصميم.

In [ ]:
class Dog:
    """قالب (Class) يصف أي كلب."""
    def __init__(self, name: str, breed: str):
        self.name = name
        self.breed = breed

    def bark(self) -> str:
        return f"{self.name} يقول: واق واق!"

# إنشاء كائنات (Objects / Instances) من نفس القالب
dog1 = Dog("ريكس", "جيرمن شيبرد")
dog2 = Dog("لونا", "بودل")

print(dog1.bark())  # ريكس يقول: واق واق!
print(dog2.bark())  # لونا يقول: واق واق!
print(dog1.name)    # ريكس
print(dog2.breed)   # بودل

> 💡 **نصيحة الخبراء**: اصطلاح تسمية الفئات في Python هو **PascalCase** (كل كلمة تبدأ بحرف كبير، بدون شرطات سفلية)، بخلاف المتغيرات والدوال التي تستخدم `snake_case`. مثال: `class StudentRecord:` وليس `class student_record:`.

### كل شيء في Python هو كائن!

In [ ]:
x = 5
print(type(x))        # <class 'int'>

name = "Ahmed"
print(type(name))     # <class 'str'>

my_list = [1, 2, 3]
print(type(my_list))  # <class 'list'>

هذا يكشف حقيقة جوهرية: `int`, `str`, `list` نفسها هي **Classes** مبنية داخل Python! عندما تكتب `x = 5`، أنت فعلياً تُنشئ **Object** من فئة `int`. وعندما تكتب `my_list.append(4)`، أنت تستدعي **method** (دالة مرتبطة بالكائن) مُعرّفة داخل فئة `list`.

## 13.3 بناء أول Class: التشريح الكامل

### الصيغة الأساسية

In [ ]:
class ClassName:
    def __init__(self, parameter1, parameter2):
        self.attribute1 = parameter1
        self.attribute2 = parameter2

    def method_name(self):
        # كود يستخدم self.attribute1 وغيرها
        pass

لنبني مثالاً كاملاً ونُشرّح كل جزء:

In [ ]:
class Car:
    """تمثل سيارة بخصائصها الأساسية."""

    def __init__(self, brand: str, model: str, year: int):
        self.brand = brand
        self.model = model
        self.year = year
        self.mileage: int = 0  # قيمة ابتدائية افتراضية

    def drive(self, distance: int) -> None:
        """تُحرّك السيارة مسافة معينة، وتُحدّث العداد."""
        self.mileage += distance
        print(f"🚗 قطعت {self.brand} {distance} كم. المسافة الكلية: {self.mileage} كم")

    def display_info(self) -> str:
        return f"{self.year} {self.brand} {self.model} - {self.mileage} كم"

my_car = Car("تويوتا", "كورولا", 2023)
my_car.drive(150)
my_car.drive(80)
print(my_car.display_info())

**الناتج:**

```
🚗 قطعت تويوتا 150 كم. المسافة الكلية: 150 كم
🚗 قطعت تويوتا 80 كم. المسافة الكلية: 230 كم
2023 تويوتا كورولا - 230 كم
```

### تشريح `__init__`: دالة البناء (Constructor)

`__init__` هي دالة خاصة (تُسمى **Magic Method** أو **Dunder Method** — اختصار لـ "Double Underscore") تُستدعى **تلقائياً** عند إنشاء كائن جديد من الفئة.

In [ ]:
my_car = Car("تويوتا", "كورولا", 2023)
# هذا يُستدعي داخلياً:
# Car.__init__(my_car, "تويوتا", "كورولا", 2023)

> 💡 **نصيحة الخبراء**: `__init__` ليست «دالة بناء» بالمعنى الحرفي الصارم (في لغات أخرى، الـ Constructor الحقيقي هو `__new__` الذي ينشئ الكائن فعلياً في الذاكرة)؛ `__init__` تُهيّئ الكائن **بعد** إنشائه. لكن عملياً وتعليمياً، التعامل معها كـ«دالة البناء» كافٍ تماماً في 99% من الحالات، ولن نحتاج `__new__` إلا في حالات متقدمة جداً خارج نطاق هذا الكتاب.

### فهم `self` بعمق

`self` هو المعامل الأول في كل method داخل الفئة، ويُمثّل **الكائن نفسه** الذي استُدعيت منه الدالة.

In [ ]:
class Counter:
    def __init__(self):
        self.count = 0

    def increment(self):
        self.count += 1

c1 = Counter()
c2 = Counter()

c1.increment()
c1.increment()
c2.increment()

print(c1.count)  # 2
print(c2.count)  # 1  -- مستقل تماماً عن c1!

**ماذا يحدث فعلياً عند `c1.increment()`؟** Python تُترجمها داخلياً إلى:

In [ ]:
Counter.increment(c1)

أي أن `self` داخل `increment` تُصبح `c1` بالضبط. هذا هو السبب في أن `self.count += 1` تُعدّل بيانات `c1` فقط، وليس `c2`.

> ⚠️ **تحذير شائع جداً عند المبتدئين**: نسيان `self` كمعامل أول في تعريف method:

```python
class Counter:
    def increment():  # ❌ نسيان self!
        pass

c = Counter()
c.increment()  
# 💥 TypeError: increment() takes 0 positional arguments but 1 was given
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

السبب: Python تُمرر الكائن تلقائياً كأول معامل في أي استدعاء method، بغض النظر عن كتابتك له أم لا. إذا لم تستقبله بمعامل `self`، سيحدث تعارض في عدد المعاملات.

> 💡 **نصيحة الخبراء**: اسم `self` هو **اصطلاح (Convention)** وليس كلمة مفتاحية حقيقية في Python — يمكنك تسميته أي اسم آخر تقنياً، لكن **لا تفعل ذلك أبداً**. كل مبرمج Python في العالم يتوقع رؤية `self`، وتغييره يجعل كودك غريباً ومربكاً لأي شخص آخر يقرأه.

### Attributes مقابل Methods

| المصطلح | الوصف | مثال |
|---|---|---|
| **Attribute (خاصية)** | بيانات مُخزّنة داخل الكائن | `self.name`, `self.age` |
| **Method (طريقة/تابع)** | دالة مُعرّفة داخل الفئة، تعمل على بيانات الكائن | `def drive(self):` |

In [ ]:
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width    # attribute
        self.height = height  # attribute

    def area(self) -> float:  # method
        return self.width * self.height

    def perimeter(self) -> float:  # method
        return 2 * (self.width + self.height)

rect = Rectangle(5, 3)
print(rect.width)        # 5 - الوصول لـ attribute (بدون أقواس)
print(rect.area())       # 15.0 - استدعاء method (مع أقواس)

## 13.4 Instance Attributes مقابل Class Attributes

### Instance Attributes (خصائص الكائن)

هذه ما رأيناه حتى الآن: خصائص خاصة بكل كائن (Instance) على حدة، تُعرَّف عادة داخل `__init__` باستخدام `self.`.

### Class Attributes (خصائص الفئة)

خصائص **مشتركة بين كل الكائنات** المُنشأة من نفس الفئة، تُعرَّف مباشرة داخل الفئة (خارج أي method).

In [ ]:
class Dog:
    species = "Canis familiaris"  # Class Attribute - مشتركة بين كل الكلاب

    def __init__(self, name: str):
        self.name = name  # Instance Attribute - خاصة بكل كلب

    def __str__(self) -> str:
        return f"{self.name} ({self.species})"

dog1 = Dog("ريكس")
dog2 = Dog("لونا")

print(dog1.species)  # Canis familiaris
print(dog2.species)  # Canis familiaris -- نفس القيمة للجميع

# الوصول من الفئة مباشرة أيضاً ممكن
print(Dog.species)   # Canis familiaris

> ⚠️ **تحذير حرج - فخ تعديل Class Attribute من خلال Instance**:

In [ ]:
dog1.species = "Wolf"  # ⚠️ هذا لا يُعدّل Class Attribute!
                        # بل يُنشئ Instance Attribute جديد خاص بـ dog1 فقط!

print(dog1.species)  # Wolf (الآن Instance Attribute خاص به)
print(dog2.species)  # Canis familiaris (لم يتأثر إطلاقاً!)
print(Dog.species)   # Canis familiaris (الفئة نفسها لم تتغير)

لتعديل Class Attribute فعلياً بحيث يؤثر على كل الكائنات، يجب تعديلها عبر اسم الفئة مباشرة: `Dog.species = "Wolf"`.

> 💡 **نصيحة الخبراء - متى تستخدم Class Attributes؟** عندما تكون القيمة **ثابتة ومشتركة منطقياً** بين كل الكائنات، مثل: عدد الكائنات المُنشأة، قيمة ثابتة فيزيائية، إعدادات افتراضية عامة.

In [ ]:
class Employee:
    company_name = "شركة التقنية المتقدمة"  # مشتركة للجميع
    total_employees = 0  # عداد مشترك

    def __init__(self, name: str, salary: float):
        self.name = name
        self.salary = salary
        Employee.total_employees += 1  # تعديل صحيح عبر اسم الفئة

emp1 = Employee("أحمد", 8000)
emp2 = Employee("سارة", 9500)
emp3 = Employee("عمر", 7500)

print(f"عدد الموظفين: {Employee.total_employees}")  # 3
print(f"الشركة: {emp1.company_name}")  # شركة التقنية المتقدمة

> 🔄 **مقارنة**: Class Attributes تشبه «المتغيرات الساكنة» (Static Variables) في لغات مثل Java و++C — بيانات تنتمي للفئة نفسها وليس لنسخة معينة منها.

## 13.5 Methods بأنواعها الثلاثة

### Instance Methods (الأكثر شيوعاً)

كل ما رأيناه حتى الآن — دوال تأخذ `self` وتعمل على بيانات كائن معين.

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float = 0):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:  # instance method
        self.balance += amount

### Class Methods: تعمل على الفئة نفسها

In [ ]:
class Employee:
    company_name = "شركة التقنية"
    total_employees = 0

    def __init__(self, name: str, salary: float):
        self.name = name
        self.salary = salary
        Employee.total_employees += 1

    @classmethod
    def get_total_employees(cls) -> int:
        """تستقبل cls (الفئة نفسها) بدلاً من self (كائن محدد)."""
        return cls.total_employees

    @classmethod
    def from_string(cls, data_string: str) -> "Employee":
        """Alternative Constructor - نمط شائع جداً لإنشاء كائنات من صيغ بيانات مختلفة."""
        name, salary = data_string.split(",")
        return cls(name.strip(), float(salary.strip()))

emp1 = Employee("أحمد", 8000)
emp2 = Employee.from_string("سارة, 9500")  # إنشاء من نص!

print(emp2.name, emp2.salary)              # سارة 9500.0
print(Employee.get_total_employees())      # 2

> 💡 **نصيحة الخبراء - نمط Alternative Constructor**: استخدام `@classmethod` لإنشاء «طرق بديلة» لإنشاء كائنات (مثل `from_string`, `from_dict`, `from_json`) هو نمط احترافي شائع جداً في مكتبات Python الكبرى. بدلاً من كتابة منطق تحليل البيانات خارج الفئة، تجعله جزءاً من الفئة نفسها بوضوح.

### Static Methods: لا تحتاج self ولا cls

In [ ]:
class MathHelper:
    @staticmethod
    def is_prime(n: int) -> bool:
        """دالة مساعدة لا تحتاج الوصول لأي بيانات خاصة بالفئة أو الكائن."""
        if n < 2:
            return False
        for i in range(2, int(n ** 0.5) + 1):
            if n % i == 0:
                return False
        return True

# تُستدعى عبر الفئة مباشرة، بدون إنشاء كائن!
print(MathHelper.is_prime(17))  # True
print(MathHelper.is_prime(18))  # False

> 💡 **نصيحة الخبراء - متى نستخدم static method؟** عندما تكون الدالة منطقياً **مرتبطة** بموضوع الفئة، لكنها **لا تحتاج** الوصول لـ `self` (بيانات كائن محدد) ولا `cls` (بيانات الفئة نفسها). إنها ببساطة دالة عادية «مُنظَّمة» داخل الفئة لأسباب تنظيمية ودلالية.

### جدول مقارنة الأنواع الثلاثة

| النوع | Decorator | المعامل الأول | متى تستخدمه |
|---|---|---|---|
| **Instance Method** | (لا يوجد) | `self` | السلوك الطبيعي الذي يعمل على بيانات كائن محدد |
| **Class Method** | `@classmethod` | `cls` | التعامل مع بيانات الفئة، أو Alternative Constructors |
| **Static Method** | `@staticmethod` | (لا شيء) | دالة مساعدة منطقياً مرتبطة، لا تحتاج self/cls |

In [ ]:
class Pizza:
    def __init__(self, toppings: list[str]):
        self.toppings = toppings

    def instance_method(self):
        return f"هذه بيتزا بإضافات: {self.toppings}"

    @classmethod
    def margherita(cls) -> "Pizza":
        return cls(["جبنة", "طماطم"])  # Alternative constructor

    @staticmethod
    def calculate_cooking_time(size_cm: int) -> int:
        return size_cm // 2  # منطق عام لا يحتاج بيانات بيتزا معينة

p1 = Pizza.margherita()
print(p1.toppings)  # ['جبنة', 'طماطم']
print(Pizza.calculate_cooking_time(30))  # 15

## 13.6 Magic Methods (Dunder Methods): تخصيص سلوك الكائن

### `__str__` و `__repr__`: كيف يُطبع الكائن؟

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

p = Point(3, 4)
print(p)  # <__main__.Point object at 0x7f8a1c0b3d90>  -- غير مفيد!

بدون تخصيص، طباعة الكائن تُظهر عنوان الذاكرة فقط — غير مفيد إطلاقاً. نُصلح هذا بـ `__str__`:

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __str__(self) -> str:
        """يُستدعى عند print() أو str() - للعرض البشري المقروء."""
        return f"نقطة({self.x}, {self.y})"

    def __repr__(self) -> str:
        """يُستدعى في REPL أو داخل قوائم - للعرض التقني الدقيق."""
        return f"Point(x={self.x}, y={self.y})"

p = Point(3, 4)
print(p)       # نقطة(3, 4)  -- يستخدم __str__
print([p])     # [Point(x=3, y=4)]  -- القوائم تستخدم __repr__ للعناصر
print(repr(p)) # Point(x=3, y=4)

> 💡 **نصيحة الخبراء - القاعدة الذهبية**: `__str__` للمستخدم النهائي (واضح ومقروء)، `__repr__` للمطور (دقيق وغالباً قابل لإعادة إنشاء الكائن منه). **الأفضل دائماً**: عرّف `__repr__` على الأقل؛ إذا لم تُعرّف `__str__`، Python تستخدم `__repr__` كبديل تلقائي.

### `__eq__`: المقارنة بين الكائنات

In [ ]:
class Point:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __eq__(self, other) -> bool:
        if not isinstance(other, Point):
            return False
        return self.x == other.x and self.y == other.y

p1 = Point(3, 4)
p2 = Point(3, 4)
p3 = Point(5, 6)

print(p1 == p2)  # True  -- بدون __eq__ كانت ستكون False (مقارنة مراجع الذاكرة)!
print(p1 == p3)  # False

> ⚠️ **تحذير شائع**: بدون تخصيص `__eq__`، عامل `==` يقارن **عناوين الذاكرة** (هل هما نفس الكائن فعلياً؟) وليس محتوى البيانات. هذا سبب شائع لحيرة المبتدئين: "لماذا كائنان متطابقان في البيانات لكن `==` تُعطي False؟"

### `__len__`: دعم دالة len()

In [ ]:
class Playlist:
    def __init__(self, songs: list[str]):
        self.songs = songs

    def __len__(self) -> int:
        return len(self.songs)

playlist = Playlist(["أغنية 1", "أغنية 2", "أغنية 3"])
print(len(playlist))  # 3  -- بدون __len__ ستحصل على TypeError

### `__add__` وتحميل العوامل (Operator Overloading)

In [ ]:
class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    def __add__(self, other: "Vector") -> "Vector":
        return Vector(self.x + other.x, self.y + other.y)

    def __repr__(self) -> str:
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)
v3 = v1 + v2  # يستدعي v1.__add__(v2) تلقائياً!
print(v3)  # Vector(4, 6)

### جدول مرجعي لأهم Magic Methods

| Magic Method | يُستدعى عند | مثال |
|---|---|---|
| `__init__` | إنشاء كائن جديد | `Point(3, 4)` |
| `__str__` | `print()`, `str()` | `print(p)` |
| `__repr__` | عرض تقني، REPL | `repr(p)`, `[p]` |
| `__eq__` | `==` | `p1 == p2` |
| `__lt__` | `<` | `p1 < p2` |
| `__len__` | `len()` | `len(playlist)` |
| `__add__` | `+` | `v1 + v2` |
| `__getitem__` | `obj[index]` | `playlist[0]` |
| `__contains__` | `in` | `"x" in obj` |
| `__call__` | استدعاء الكائن كدالة | `obj()` |

> 🔄 **مقارنة مع لغات أخرى**: هذا المفهوم يُسمى **Operator Overloading**، وهو موجود في ++C وJava (بصيغ مختلفة) وغيرها. في Python، آلية التنفيذ أنظف وأوضح بكثير بفضل الـ Dunder Methods الموحدة.

## 13.7 خصوصية البيانات: Public, Protected, Private (بأسلوب Python)

### فلسفة Python: "We're all consenting adults here"

على عكس لغات مثل Java التي تفرض `private`/`public`/`protected` بشكل صارم على مستوى اللغة، Python تعتمد على **اصطلاحات التسمية** (Naming Conventions) بدلاً من قيود حقيقية.

In [ ]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner           # Public - وصول حر من أي مكان
        self._balance = balance      # Protected (اصطلاحاً) - "استخدم بحذر"
        self.__pin = "1234"          # Private (اصطلاحاً) - "لا تلمس من خارج الفئة"

| النمط | الاسم | المعنى |
|---|---|---|
| `self.name` | Public | وصول حر تماماً، جزء من الواجهة العامة للفئة |
| `self._name` | Protected (اصطلاحي) | «علامة تحذير» للمطورين: هذا تفصيل داخلي، لا تعتمد عليه من خارج الفئة |
| `self.__name` | Private (شبه حقيقي) | يخضع لـ Name Mangling (انظر أدناه) |

### Single Underscore: `_balance`

In [ ]:
class BankAccount:
    def __init__(self, balance: float):
        self._balance = balance  # "تلميح" أنه داخلي، لكن لا شيء يمنع الوصول فعلياً

account = BankAccount(1000)
print(account._balance)  # 1000 -- يعمل! Python لا تمنعك، لكنها تُعتبر ممارسة سيئة

> 💡 **نصيحة الخبراء**: `_balance` بشرطة سفلية واحدة هي **اتفاق بين المبرمجين**، وليست قيداً تقنياً حقيقياً. معناها الضمني: «هذا تفصيل تنفيذي داخلي، قد يتغير في أي وقت، فلا تعتمد عليه مباشرة من خارج الفئة — استخدم methods عامة بدلاً من ذلك.»

### Double Underscore: `__pin` و Name Mangling

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    class BankAccount:
        def __init__(self, pin: str):
            self.__pin = pin

        def verify_pin(self, attempt: str) -> bool:
            return self.__pin == attempt

    account = BankAccount("1234")
    print(account.verify_pin("1234"))  # True

    # print(account.__pin)  
    # 💥 AttributeError: 'BankAccount' object has no attribute '__pin'
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**ماذا حدث؟** Python تُطبّق **Name Mangling**: تُحوّل `__pin` داخلياً إلى `_BankAccount__pin`. يمكنك تقنياً الوصول إليه بهذا الاسم المُعدَّل، لكن هذا يُعتبر اختراقاً متعمداً لا يجب فعله أبداً:

In [ ]:
print(account._BankAccount__pin)  # 1234  -- يعمل تقنياً، لكن لا تفعل هذا أبداً!

> 💡 **الهدف الحقيقي من Name Mangling**: ليس «الحماية الأمنية» (فهو قابل للتجاوز كما رأينا)، بل **منع تعارض الأسماء** عند الوراثة المتعددة (سنتعمق في الوراثة في الفصل 14).

### Getters و Setters بأسلوب Python: `@property`

في لغات مثل Java، من الشائع كتابة `getBalance()` و`setBalance()` صراحة. في Python، نستخدم `@property` لإعطاء نفس الحماية لكن بواجهة أنظف تبدو كخاصية عادية:

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    class BankAccount:
        def __init__(self, balance: float):
            self._balance = balance

        @property
        def balance(self) -> float:
            """Getter - تُستدعى عند قراءة account.balance"""
            return self._balance

        @balance.setter
        def balance(self, value: float) -> None:
            """Setter - تُستدعى عند account.balance = value"""
            if value < 0:
                raise ValueError("الرصيد لا يمكن أن يكون سالباً!")
            self._balance = value

    account = BankAccount(1000)
    print(account.balance)   # 1000 -- يبدو كخاصية عادية، لكنه method بالخلف!

    account.balance = 1500   # يستدعي الـ setter تلقائياً
    print(account.balance)   # 1500

    # account.balance = -500  
    # 💥 ValueError: الرصيد لا يمكن أن يكون سالباً!
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

> 💡 **نصيحة الخبراء - لماذا هذا أفضل من getter/setter التقليدية؟** يمكنك البدء بخاصية `public` عادية (`self.balance`)، ثم إذا احتجت لاحقاً إضافة منطق تحقق (Validation)، تُحوّلها لـ `@property` **دون تغيير** الكود الذي يستخدمها في أي مكان آخر (`account.balance` يبقى يعمل بنفس الصيغة). هذا مثال رائع على مبدأ **«الواجهة المستقرة، التنفيذ الداخلي مرن»**.

سنتعمق أكثر في `@property` والـ Dataclasses في **الفصل الرابع عشر**.

## 13.8 مشروع تطبيقي: نظام إدارة مكتبة شخصية

```python
class Book:
    """يمثل كتاباً واحداً في المكتبة."""

    total_books = 0  # Class Attribute - عداد لكل الكتب المُنشأة

    def __init__(self, title: str, author: str, pages: int, is_read: bool = False):
        self.title = title
        self.author = author
        self.pages = pages
        self._is_read = is_read
        Book.total_books += 1

    @property
    def is_read(self) -> bool:
        return self._is_read

    def mark_as_read(self) -> None:
        self._is_read = True
        print(f"✅ '{self.title}' تم وضع علامة 'مقروء' عليه.")

    def __str__(self) -> str:
        status = "📖 مقروء" if self._is_read else "📕 غير مقروء"
        return f"{self.title} - {self.author} ({self.pages} صفحة) [{status}]"

    def __repr__(self) -> str:
        return f"Book(title='{self.title}', author='{self.author}')"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Book):
            return False
        return self.title == other.title and self.author == other.author

    @classmethod
    def from_string(cls, data: str) -> "Book":
        """ينشئ كتاباً من نص بصيغة: 'العنوان|المؤلف|الصفحات'"""
        title, author, pages = data.split("|")
        return cls(title.strip(), author.strip(), int(pages.strip()))

    @staticmethod
    def estimate_reading_hours(pages: int, pages_per_hour: int = 30) -> float:
        """تقدير عدد الساعات اللازمة لقراءة كتاب بعدد صفحات معين."""
        return round(pages / pages_per_hour, 1)

class Library:
    """تمثل مكتبة تحتوي على مجموعة من الكتب."""

    def __init__(self, name: str):
        self.name = name
        self.books: list[Book] = []

    def add_book(self, book: Book) -> None:
        if book in self.books:
            print(f"⚠️ '{book.title}' موجود مسبقاً في المكتبة.")
            return
        self.books.append(book)
        print(f"✅ تمت إضافة '{book.title}' إلى مكتبة '{self.name}'.")

    def remove_book(self, title: str) -> None:
        for book in self.books:
            if book.title == title:
                self.books.remove(book)
                print(f"🗑️ تم حذف '{title}'.")
                return
        print(f"❌ لم يُعثر على '{title}'.")

    def get_unread_books(self) -> list[Book]:
        return [book for book in self.books if not book.is_read]

    def total_pages(self) -> int:
        return sum(book.pages for book in self.books)

    def find_by_author(self, author: str) -> list[Book]:
        return [book for book in self.books if book.author == author]

    def __len__(self) -> int:
        return len(self.books)

    def __str__(self) -> str:
        return f"مكتبة '{self.name}' - {len(self.books)} كتاب"

    def print_all(self) -> None:
        print(f"\n📚 {self}")
        print("-" * 50)
        for i, book in enumerate(self.books, start=1):
            print(f"{i}. {book}")
        print("-" * 50)

# --- التشغيل التجريبي ---
if __name__ == "__main__":
    my_library = Library("مكتبة أحمد")

    book1 = Book("نادي السيارات", "خالد عبدالله", 250)
    book2 = Book.from_string("الخيميائي | باولو كويلو | 180")
    book3 = Book("Clean Code", "Robert Martin", 400, is_read=True)

    my_library.add_book(book1)
    my_library.add_book(book2)
    my_library.add_book(book3)
    my_library.add_book(book1)  # محاولة إضافة مكررة

    my_library.print_all()

    book1.mark_as_read()

    print(f"\n📖 عدد الكتب غير المقروءة: {len(my_library.get_unread_books())}")
    print(f"📄 إجمالي الصفحات: {my_library.total_pages()}")
    print(f"📊 إجمالي الكتب في كل المكتبات: {Book.total_books}")

    hours = Book.estimate_reading_hours(book2.pages)
    print(f"⏱️ وقت قراءة متوقع لـ '{book2.title}': {hours} ساعة")

    print(f"\n{my_library}")  # يستخدم __str__
    print(f"عدد الكتب (len): {len(my_library)}")  # يستخدم __len__
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**الناتج المتوقع:**

```
✅ تمت إضافة 'نادي السيارات' إلى مكتبة 'مكتبة أحمد'.
✅ تمت إضافة 'الخيميائي' إلى مكتبة 'مكتبة أحمد'.
✅ تمت إضافة 'Clean Code' إلى مكتبة 'مكتبة أحمد'.
⚠️ 'نادي السيارات' موجود مسبقاً في المكتبة.

📚 مكتبة 'مكتبة أحمد' - 3 كتاب
--------------------------------------------------
1. نادي السيارات - خالد عبدالله (250 صفحة) [📕 غير مقروء]
2. الخيميائي - باولو كويلو (180 صفحة) [📕 غير مقروء]
3. Clean Code - Robert Martin (400 صفحة) [📖 مقروء]
--------------------------------------------------
✅ 'نادي السيارات' تم وضع علامة 'مقروء' عليه.

📖 عدد الكتب غير المقروءة: 1
📄 إجمالي الصفحات: 830
📊 إجمالي الكتب في كل المكتبات: 3
⏱️ وقت قراءة متوقع لـ 'الخيميائي': 6.0 ساعة

مكتبة 'مكتبة أحمد' - 3 كتاب
عدد الكتب (len): 3
```

<details>
<summary>✅ تحدي إضافي: إضافة تصنيف وترتيب حسب الصفحات (الحل)</summary>

```python
def sort_by_pages(self, reverse: bool = False) -> list[Book]:
    """ترتيب كتب المكتبة حسب عدد الصفحات."""
    return sorted(self.books, key=lambda book: book.pages, reverse=reverse)

# إضافة هذا كـ method داخل Library، ثم:
sorted_books = my_library.sort_by_pages()
for book in sorted_books:
    print(f"{book.title}: {book.pages} صفحة")
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Book`، `my_library` من مثال سابق.

</details>

## 13.9 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1**: أنشئ فئة `Person` مع `__init__` يستقبل `name` و`age`، وmethod باسم `introduce()` تطبع جملة تعارف.

**تمرين 2**: أضف Class Attribute باسم `species = "Homo sapiens"` لفئة `Person`، واطبعه من كائنين مختلفين.

<details>
<summary>✅ الحل</summary>

In [ ]:
class Person:
    species = "Homo sapiens"

    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def introduce(self) -> None:
        print(f"مرحباً، اسمي {self.name} وعمري {self.age} سنة.")

p1 = Person("خالد", 30)
p2 = Person("منى", 25)

p1.introduce()  # مرحباً، اسمي خالد وعمري 30 سنة.
p2.introduce()  # مرحباً، اسمي منى وعمري 25 سنة.

print(p1.species)  # Homo sapiens
print(p2.species)  # Homo sapiens

</details>

### 🟡 مستوى متوسط

**تمرين 3**: أنشئ فئة `Rectangle` تحتوي `width`, `height`، وmethods: `area()`, `perimeter()`, و`__str__` تعرض الأبعاد والمساحة. أضف `__eq__` لمقارنة مستطيلين بناءً على المساحة.

**تمرين 4**: أنشئ فئة `TemperatureConverter` باستخدام `@staticmethod` لدالتين: `celsius_to_fahrenheit(c)` و`fahrenheit_to_celsius(f)`.

<details>
<summary>✅ الحل</summary>

In [ ]:
# تمرين 3
class Rectangle:
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

    def __str__(self) -> str:
        return f"مستطيل {self.width}x{self.height}, المساحة={self.area()}"

    def __eq__(self, other) -> bool:
        if not isinstance(other, Rectangle):
            return False
        return self.area() == other.area()

r1 = Rectangle(4, 5)
r2 = Rectangle(2, 10)
print(r1)              # مستطيل 4x5, المساحة=20
print(r1 == r2)         # True -- نفس المساحة رغم اختلاف الأبعاد!

# تمرين 4
class TemperatureConverter:
    @staticmethod
    def celsius_to_fahrenheit(c: float) -> float:
        return (c * 9/5) + 32

    @staticmethod
    def fahrenheit_to_celsius(f: float) -> float:
        return (f - 32) * 5/9

print(TemperatureConverter.celsius_to_fahrenheit(100))  # 212.0
print(TemperatureConverter.fahrenheit_to_celsius(32))   # 0.0

</details>

### 🔴 تحدي

**تمرين 5**: أنشئ فئة `ShoppingCart` تحتوي:
- قائمة داخلية من المنتجات (كل منتج قاموس فيه `name` و`price` و`quantity`)
- method لإضافة منتج
- method لحساب الإجمالي
- `@property` باسم `item_count` تُرجع عدد القطع الكلي
- `__str__` تعرض محتويات السلة بشكل منسّق
- `__len__` تُرجع عدد المنتجات المختلفة (وليس الكمية الكلية)

<details>
<summary>✅ الحل</summary>

In [ ]:
class ShoppingCart:
    def __init__(self):
        self._items: list[dict] = []

    def add_item(self, name: str, price: float, quantity: int = 1) -> None:
        self._items.append({"name": name, "price": price, "quantity": quantity})
        print(f"✅ أُضيف {quantity}x {name} إلى السلة.")

    def total_price(self) -> float:
        return sum(item["price"] * item["quantity"] for item in self._items)

    @property
    def item_count(self) -> int:
        """إجمالي عدد القطع (مجموع الكميات)."""
        return sum(item["quantity"] for item in self._items)

    def __len__(self) -> int:
        """عدد المنتجات المختلفة في السلة."""
        return len(self._items)

    def __str__(self) -> str:
        lines = [f"🛒 سلة التسوق ({len(self)} منتج مختلف):"]
        for item in self._items:
            subtotal = item["price"] * item["quantity"]
            lines.append(f"  - {item['name']}: {item['quantity']} × {item['price']} = {subtotal}")
        lines.append(f"الإجمالي: {self.total_price()} ر.س")
        return "\n".join(lines)

cart = ShoppingCart()
cart.add_item("لابتوب", 3500, 1)
cart.add_item("ماوس", 50, 2)
cart.add_item("كيبورد", 150, 1)

print(cart)
print(f"\nعدد القطع الكلي: {cart.item_count}")  # 4
print(f"عدد المنتجات المختلفة: {len(cart)}")    # 3

</details>

## 13.10 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: ما الفرق بين Class و Object؟</summary>

**الإجابة**: Class هو القالب/المخطط الذي يصف الخصائص والسلوك، بينما Object هو نسخة فعلية (Instance) مُنشأة من ذلك القالب، تحمل بيانات حقيقية خاصة بها. يمكن إنشاء عدة Objects مستقلة من نفس Class.
</details>

<details>
<summary>❓ السؤال 2: لماذا يجب أن يكون self أول معامل في كل instance method؟</summary>

**الإجابة**: لأن Python تُمرر الكائن نفسه تلقائياً كأول معامل عند استدعاء أي method عبر `object.method()`. إذا لم تستقبل الدالة هذا المعامل، سيحدث تعارض في عدد المعاملات (TypeError). `self` يُمكّن الـ method من الوصول لبيانات ذلك الكائن المحدد بالذات.
</details>

<details>
<summary>❓ السؤال 3: ما الفرق بين Instance Attribute و Class Attribute؟</summary>

**الإجابة**: Instance Attribute خاص بكل كائن على حدة (يُعرَّف عادة بـ `self.x` داخل `__init__`)، بينما Class Attribute مشترك بين كل الكائنات المُنشأة من نفس الفئة (يُعرَّف مباشرة داخل جسم الفئة، خارج أي method). تعديل Instance Attribute لا يؤثر على الكائنات الأخرى، بينما تعديل Class Attribute (عبر اسم الفئة) يؤثر على الجميع.
</details>

<details>
<summary>❓ السؤال 4: متى تستخدم @staticmethod بدلاً من @classmethod أو instance method عادية؟</summary>

**الإجابة**: تستخدم `@staticmethod` عندما تكون الدالة منطقياً مرتبطة بموضوع الفئة، لكنها لا تحتاج الوصول لا لبيانات كائن محدد (`self`) ولا لبيانات الفئة نفسها (`cls`). إنها دالة مستقلة «مُنظَّمة» داخل الفئة فقط لأسباب دلالية وتنظيمية.
</details>

<details>
<summary>❓ السؤال 5: لماذا نستخدم @property بدلاً من الوصول المباشر لـ attribute؟</summary>

**الإجابة**: `@property` تسمح بإضافة منطق تحقق (Validation) أو حساب عند قراءة/تعديل قيمة، مع الحفاظ على واجهة استخدام بسيطة تبدو كخاصية عادية (`obj.value` بدلاً من `obj.get_value()`). هذا يسمح بالبدء بخاصية بسيطة ثم تطويرها لاحقاً بإضافة تحققات دون تغيير الكود الذي يستخدمها في أي مكان آخر بالمشروع.
</details>

## 13.11 ملخص الفصل (Cheat Sheet)

```python
# تعريف فئة أساسية
class ClassName:
    class_attribute = "مشترك بين الجميع"  # Class Attribute

    def __init__(self, param1, param2):
        self.attr1 = param1    # Instance Attribute
        self.attr2 = param2

    def instance_method(self):          # self = الكائن نفسه
        return self.attr1

    @classmethod
    def class_method(cls):              # cls = الفئة نفسها
        return cls.class_attribute

    @staticmethod
    def static_method():                # لا self ولا cls
        return "دالة مستقلة منطقياً"

# إنشاء كائن
obj = ClassName("value1", "value2")

# Magic Methods الأساسية
class MyClass:
    def __init__(self, x): self.x = x
    def __str__(self): return f"قابل للقراءة: {self.x}"
    def __repr__(self): return f"MyClass({self.x})"
    def __eq__(self, other): return self.x == other.x
    def __len__(self): return self.x
    def __add__(self, other): return MyClass(self.x + other.x)

# خصوصية البيانات (اصطلاحية)
self.public_attr      # وصول حر
self._protected_attr  # "تحذير" - داخلي، لا تعتمد عليه من الخارج
self.__private_attr   # Name Mangling → _ClassName__private_attr

# Property - getter/setter أنيق
class Account:
    def __init__(self, balance):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("لا يمكن أن يكون سالباً")
        self._balance = value
```

🧾 **للقراءة فقط** — `self` لا وجود له خارج الكائن؛ هذه بطاقة مرجعية.

| المفهوم | القاعدة الذهبية |
|---|---|
| تسمية الفئات | PascalCase: `class StudentRecord:` |
| self | يجب أن يكون أول معامل في كل instance method |
| Class Attribute | للبيانات المشتركة فعلاً بين كل الكائنات |
| `__str__` | دائماً أضفها للعرض المقروء البشري |
| `_` و `__` | اصطلاحات فقط، وليست حماية حقيقية صارمة |
| `@property` | الخيار الافتراضي الاحترافي بدلاً من get/set يدوية |

## 13.12 ماذا بعد؟

لقد بنيت الآن أساساً متيناً في البرمجة الكائنية: تعرف كيف تُصمم فئة، تُنشئ كائنات، تُخصص سلوكها عبر Magic Methods، وتتحكم في خصوصية بياناتها.

لكن هناك سؤال جوهري لم نُجب عليه بعد: ماذا لو أردت إنشاء فئة `Dog` وفئة `Cat`، وكلاهما «حيوان» يشترك في خصائص كثيرة (اسم، عمر، صوت) لكن لكل منهما سلوك خاص؟ هل تُكرر نفس الكود في الفئتين؟ بالطبع لا!

هذا هو موضوع **الوراثة (Inheritance)** — أحد الأعمدة الأربعة للبرمجة الكائنية. في **الفصل الرابع عشر: البرمجة الكائنية - التعمق**، سنغوص في الوراثة، وتعدد الأشكال (Polymorphism)، والـ Dataclasses التي تُبسّط كتابة الفئات بشكل كبير، وسنختم بمشروع متكامل: **نظام حجز مكتبة** يجمع كل مفاهيم الفصلين 13 و14 معاً.

---

### الخطوة التالية

- [الفصل الرابع عشر: البرمجة الكائنية](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-14.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الثالث عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*